<a href="https://colab.research.google.com/github/iescobedo5/Computer-Vision/blob/main/new_SSD_7_5k_20e.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#mount drive to use dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q /content/drive/MyDrive/subset7_5k.zip -d /content/subset7_5k

In [ ]:
import os
import time
import json
import tempfile
from pathlib import Path

import torch
import torchvision
import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision.models.detection import ssd300_vgg16
from torchvision.models.detection.ssd import SSDClassificationHead

In [ ]:
!pip install pycocotools

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

In [ ]:
DATA_ROOT = Path("/content/subset7_5k/content/UADETRAC/DETRAC_Upload/data/subsets/subset7_5k")

TRAIN_IMG_DIR = DATA_ROOT / "images" / "train"
TRAIN_LBL_DIR = DATA_ROOT / "labels" / "train"
VAL_IMG_DIR   = DATA_ROOT / "images" / "val"
VAL_LBL_DIR   = DATA_ROOT / "labels" / "val"

SAVE_DIR = Path("/content/drive/MyDrive/UA/ssd_results/subset7_5k_20e")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = SAVE_DIR / "ssd_training_log.csv"
SUMMARY_PATH = SAVE_DIR / "ssd_final_metrics.csv"

In [ ]:
NUM_CLASSES = 2
NUM_EPOCHS = 20
BATCH_SIZE = 8
NUM_WORKERS = 2
LEARNING_RATE = 0.001
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
SCORE_THRESHOLD = 0.3
IOU_THRESHOLD = 0.5

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

Using device: cuda
Torch version: 2.10.0+cu128
Torchvision version: 0.25.0+cu128


In [ ]:
class YOLODetectionDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)

        self.image_files = sorted([
            f for f in self.img_dir.iterdir()
            if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        lbl_path = self.lbl_dir / f"{img_path.stem}.txt"

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes = []
        labels = []

        if lbl_path.exists():
            with open(lbl_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue

                cls_id, x_center, y_center, bw, bh = map(float, parts)

                x_center *= w
                y_center *= h
                bw *= w
                bh *= h

                x_min = x_center - bw / 2
                y_min = y_center - bh / 2
                x_max = x_center + bw / 2
                y_max = y_center + bh / 2

                # Clamp
                x_min = max(0, min(x_min, w))
                y_min = max(0, min(y_min, h))
                x_max = max(0, min(x_max, w))
                y_max = max(0, min(y_max, h))

                if x_max <= x_min or y_max <= y_min:
                    continue

                boxes.append([x_min, y_min, x_max, y_max])

                labels.append(int(cls_id) + 1)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        area = torch.zeros((boxes.shape[0],), dtype=torch.float32)
        if boxes.shape[0] > 0:
            area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

        iscrowd = torch.zeros((boxes.shape[0],), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd
        }

        img = F.to_tensor(img)
        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
train_dataset = YOLODetectionDataset(TRAIN_IMG_DIR, TRAIN_LBL_DIR)
val_dataset   = YOLODetectionDataset(VAL_IMG_DIR, VAL_LBL_DIR)

print("Train images:", len(train_dataset))
print("Val images:", len(val_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

val_loader_bs1 = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

Train images: 7500
Val images: 1500


In [ ]:
def build_ssd_model(num_classes):
    model = ssd300_vgg16(weights="DEFAULT")

    num_anchors = model.anchor_generator.num_anchors_per_location()

    in_channels_list = [512, 1024, 512, 256, 256, 256] # Known values for ssd300_vgg16

    model.head.classification_head = SSDClassificationHead(
        in_channels=in_channels_list,
        num_anchors=num_anchors,
        num_classes=num_classes
    )
    return model


model = build_ssd_model(NUM_CLASSES)
model.to(device)

Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to /root/.cache/torch/hub/checkpoints/ssd300_vgg16_coco-b556d3b4.pth


100%|██████████| 136M/136M [00:00<00:00, 179MB/s]


SSD(
  (backbone): SSDFeatureExtractorVGG(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.1
)

In [ ]:
def train_one_epoch(model, loader, optimizer, device, epoch):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, desc=f"Train Epoch {epoch}", leave=False)

    for step, (images, targets) in enumerate(pbar, start=1):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item()
        avg_loss = running_loss / step
        pbar.set_postfix(loss=f"{avg_loss:.4f}")

    return running_loss / len(loader)


@torch.no_grad()
def validate_one_epoch(model, loader, device, epoch):
    # For torchvision detection models, keep train() to get losses
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, desc=f"Val Epoch {epoch}", leave=False)

    for step, (images, targets) in enumerate(pbar, start=1):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        running_loss += losses.item()
        avg_loss = running_loss / step
        pbar.set_postfix(val_loss=f"{avg_loss:.4f}")

    return running_loss / len(loader)

In [ ]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter <= 0:
        return 0.0

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0


@torch.no_grad()
def simple_precision_recall(model, loader, device, score_threshold=0.3, iou_threshold=0.5):
    model.eval()

    TP, FP, FN = 0, 0, 0

    for images, targets in tqdm(loader, desc="Simple Precision/Recall", leave=False):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for output, target in zip(outputs, targets):
            pred_boxes = output["boxes"].cpu()
            pred_scores = output["scores"].cpu()

            keep = pred_scores >= score_threshold
            pred_boxes = pred_boxes[keep]

            gt_boxes = target["boxes"]
            matched_gt = set()

            for pred_box in pred_boxes:
                pred_box = pred_box.tolist()
                best_iou = 0.0
                best_gt_idx = -1

                for gt_idx, gt_box in enumerate(gt_boxes):
                    if gt_idx in matched_gt:
                        continue

                    iou = compute_iou(pred_box, gt_box.tolist())
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = gt_idx

                if best_iou >= iou_threshold:
                    TP += 1
                    matched_gt.add(best_gt_idx)
                else:
                    FP += 1

            FN += len(gt_boxes) - len(matched_gt)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0

    return precision, recall, TP, FP, FN

In [ ]:
def build_coco_gt(dataset):
    images = []
    annotations = []
    categories = [{"id": 1, "name": "vehicle"}]

    ann_id = 1

    for img_id in range(len(dataset)):
        img, target = dataset[img_id]
        _, h, w = img.shape

        images.append({
            "id": img_id,
            "width": w,
            "height": h,
            "file_name": dataset.image_files[img_id].name
        })

        boxes = target["boxes"]
        labels = target["labels"]

        for box, label in zip(boxes, labels):
            x_min, y_min, x_max, y_max = box.tolist()
            bw = x_max - x_min
            bh = y_max - y_min

            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": int(label.item()),
                "bbox": [x_min, y_min, bw, bh],
                "area": bw * bh,
                "iscrowd": 0
            })
            ann_id += 1

    return {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

@torch.no_grad()
def get_ssd_predictions(model, loader, device, score_threshold=0.3):
    model.eval()
    predictions = []

    image_counter = 0

    for images, _ in tqdm(loader, desc="Collecting SSD predictions", leave=False):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for output in outputs:
            boxes = output["boxes"].cpu()
            scores = output["scores"].cpu()
            labels = output["labels"].cpu()

            keep = scores >= score_threshold
            boxes = boxes[keep]
            scores = scores[keep]
            labels = labels[keep]

            for box, score, label in zip(boxes, scores, labels):
                x_min, y_min, x_max, y_max = box.tolist()
                bw = x_max - x_min
                bh = y_max - y_min

                predictions.append({
                    "image_id": image_counter,
                    "category_id": int(label.item()),
                    "bbox": [x_min, y_min, bw, bh],
                    "score": float(score.item())
                })

            image_counter += 1

    return predictions

def evaluate_ssd_coco(model, dataset, loader, device, score_threshold=0.3):
    coco_gt_dict = build_coco_gt(dataset)

    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
        json.dump(coco_gt_dict, f)
        gt_json_path = f.name

    coco_gt = COCO(gt_json_path)
    predictions = get_ssd_predictions(model, loader, device, score_threshold=score_threshold)

    if len(predictions) == 0:
        print("No predictions found. Try lowering score_threshold.")
        return None

    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
        json.dump(predictions, f)
        pred_json_path = f.name

    coco_dt = coco_gt.loadRes(pred_json_path)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    # Extract main stats
    stats = {
        "mAP50_95": float(coco_eval.stats[0]),
        "mAP50": float(coco_eval.stats[1]),
        "mAP75": float(coco_eval.stats[2]),
        "AR_max1": float(coco_eval.stats[6]),
        "AR_max10": float(coco_eval.stats[7]),
        "AR_max100": float(coco_eval.stats[8]),
    }

    return stats

In [ ]:
@torch.no_grad()
def measure_ssd_fps(model, loader, device, warmup_batches=5):
    model.eval()

    # Warm-up
    for i, (images, _) in enumerate(loader):
        images = [img.to(device) for img in images]
        _ = model(images)
        if i + 1 >= warmup_batches:
            break

    if device.type == "cuda":
        torch.cuda.synchronize()

    total_images = 0
    total_time = 0.0

    for images, _ in tqdm(loader, desc="Measuring FPS", leave=False):
        images = [img.to(device) for img in images]
        batch_size = len(images)

        if device.type == "cuda":
            torch.cuda.synchronize()
        start_time = time.time()

        _ = model(images)

        if device.type == "cuda":
            torch.cuda.synchronize()
        end_time = time.time()

        total_time += (end_time - start_time)
        total_images += batch_size

    avg_time_sec = total_time / total_images if total_images > 0 else 0.0
    fps = total_images / total_time if total_time > 0 else 0.0

    return {
        "total_images": total_images,
        "total_time_sec": total_time,
        "avg_time_per_image_sec": avg_time_sec,
        "avg_time_per_image_ms": avg_time_sec * 1000.0,
        "fps": fps
    }

In [ ]:
history = []
best_val_loss = float("inf")
best_epoch = -1

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch)
    val_loss = validate_one_epoch(model, val_loader, device, epoch)

    lr_scheduler.step()

    epoch_time_sec = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"\nEpoch [{epoch}/{NUM_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"LR:         {current_lr:.6f}")
    print(f"Time:       {epoch_time_sec/60:.2f} min")

    # Save last
    torch.save(model.state_dict(), SAVE_DIR / "ssd_last_7_5k20e.pth")

    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), SAVE_DIR / "ssd_best_7_50k20e.pth")
        print("Saved best model.")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": current_lr,
        "epoch_time_sec": epoch_time_sec,
        "epoch_time_min": epoch_time_sec / 60.0
    })

    pd.DataFrame(history).to_csv(CSV_PATH, index=False)

print("\nTraining complete.")
print(f"Best val loss: {best_val_loss:.4f} at epoch {best_epoch}")


Epoch [1/20]
Train Loss: 1.7739
Val Loss:   1.4008
LR:         0.001000
Time:       6.51 min
Saved best model.



Epoch [2/20]
Train Loss: 1.2378
Val Loss:   1.2706
LR:         0.001000
Time:       6.42 min
Saved best model.



Epoch [3/20]
Train Loss: 1.0740
Val Loss:   1.1732
LR:         0.001000
Time:       6.44 min
Saved best model.



Epoch [4/20]
Train Loss: 0.9588
Val Loss:   1.1108
LR:         0.001000
Time:       6.47 min
Saved best model.



Epoch [5/20]
Train Loss: 0.8729
Val Loss:   1.0783
LR:         0.000100
Time:       6.47 min
Saved best model.



Epoch [6/20]
Train Loss: 0.7298
Val Loss:   1.0291
LR:         0.000100
Time:       6.56 min
Saved best model.


... (10 outputs left out)


Epoch [17/20]
Train Loss: 0.6444
Val Loss:   1.0331
LR:         0.000001
Time:       6.48 min



Epoch [18/20]
Train Loss: 0.6432
Val Loss:   1.0328
LR:         0.000001
Time:       6.46 min



Epoch [19/20]
Train Loss: 0.6435
Val Loss:   1.0328
LR:         0.000001
Time:       6.53 min



Epoch [20/20]
Train Loss: 0.6429
Val Loss:   1.0328
LR:         0.000000
Time:       6.49 min

Training complete.
Best val loss: 1.0291 at epoch 6


In [ ]:
best_model = build_ssd_model(NUM_CLASSES)
best_model.load_state_dict(torch.load(SAVE_DIR / "ssd_best_7_50k20e.pth", map_location=device))
best_model.to(device)
best_model.eval()

print("\nLoaded best SSD model for evaluation.")


Loaded best SSD model for evaluation.


In [ ]:
coco_stats = evaluate_ssd_coco(
    best_model,
    val_dataset,
    val_loader,
    device,
    score_threshold=SCORE_THRESHOLD
)

# Simple precision / recall
precision, recall, TP, FP, FN = simple_precision_recall(
    best_model,
    val_loader,
    device,
    score_threshold=SCORE_THRESHOLD,
    iou_threshold=IOU_THRESHOLD
)

# FPS with batch size 1
fps_stats = measure_ssd_fps(
    best_model,
    val_loader_bs1,
    device,
    warmup_batches=5
)

print("\n================ FINAL RESULTS ================")
if coco_stats is not None:
    print(f"mAP50-95: {coco_stats['mAP50_95']:.4f}")
    print(f"mAP50:    {coco_stats['mAP50']:.4f}")
    print(f"mAP75:    {coco_stats['mAP75']:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"TP: {TP}, FP: {FP}, FN: {FN}")
print(f"Avg inference time/image: {fps_stats['avg_time_per_image_ms']:.2f} ms")
print(f"FPS: {fps_stats['fps']:.2f}")

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=2.48s).
Accumulating evaluation results...
DONE (t=0.22s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.703
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.902
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.845
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.277
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.728
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.832
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.121
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.670
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.742
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10


================ FINAL RESULTS ================
mAP50-95: 0.7025
mAP50:    0.9020
mAP75:    0.8454
Precision: 0.9423
Recall:    0.9117
TP: 9960, FP: 610, FN: 965
Avg inference time/image: 18.90 ms
FPS: 52.92


In [ ]:
summary = {
    "dataset_root": str(DATA_ROOT),
    "num_train_images": len(train_dataset),
    "num_val_images": len(val_dataset),
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "score_threshold": SCORE_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "precision": precision,
    "recall": recall,
    "TP": TP,
    "FP": FP,
    "FN": FN,
    "avg_inference_time_ms": fps_stats["avg_time_per_image_ms"],
    "fps": fps_stats["fps"],
}

if coco_stats is not None:
    summary.update(coco_stats)

summary_df = pd.DataFrame([summary])
summary_df.to_csv(SUMMARY_PATH, index=False)

print(f"\nSaved training log to: {CSV_PATH}")
print(f"Saved final summary to: {SUMMARY_PATH}")
print(f"Saved best weights to: {SAVE_DIR / 'ssd_best_7_50k20e.pth'}")
print(f"Saved last weights to: {SAVE_DIR / 'ssd_last_7_5k20e.pth'}")


Saved training log to: /content/drive/MyDrive/UA/ssd_results/subset7_5k_20e/ssd_training_log.csv
Saved final summary to: /content/drive/MyDrive/UA/ssd_results/subset7_5k_20e/ssd_final_metrics.csv
Saved best weights to: /content/drive/MyDrive/UA/ssd_results/subset7_5k_20e/ssd_best_7_50k20e.pth
Saved last weights to: /content/drive/MyDrive/UA/ssd_results/subset7_5k_20e/ssd_last_7_5k20e.pth
